# Titanic Survival Prediction - ML Pipeline

## Import Libraries

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

import joblib

# Load Dataset

In [2]:
df = pd.read_csv('train.csv')

# Remove Weak Columns and Create New ones

In [3]:
# Creating 'FamilySize' by combining SibSp and Parch(As they hold good correlation)
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1

In [4]:
# Dropping PassengerId, Name, and Cabin columns:
# -> PassengerId: A unique identifier and doesn't contribute to prediction
# -> Name: It's not directly useful without complex feature extraction
# -> Cabin: has too many missing values
# -> SibSp and Parch already combined into the 'FamilySize' feature

df = df.drop(columns = ['PassengerId', 'SibSp', 'Parch','Name', 'Ticket', 'Cabin'], axis = 1)

# Separate Feature and Target

In [5]:
X = df.drop('Survived', axis = 1)
y = df['Survived']

# Train Test Split

In [6]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 0)

# Feature Type
 - Dividing features into numerical and categorical types

 - Separating 'Sex' feature as it is a binary categorical variable,

In [7]:
numeric_features = ['Age', 'Fare', 'FamilySize']
categorical_features = ['Pclass', 'Embarked']
sex_feature = ['Sex']

# Numeric Pipeline

In [8]:
numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy = 'median')),
    ('scaler', StandardScaler())
])

# Categorical Pipeline

In [9]:
categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy = 'most_frequent')),
    ('encoder', OneHotEncoder(drop = 'first', handle_unknown = 'ignore'))
])

# sex_feature pipeline

In [10]:
sex_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy = 'most_frequent')),
    ('encoder', OrdinalEncoder())
])

# ColumnTransformer

In [11]:
preprocessor = ColumnTransformer([
    ('num', numeric_pipeline, numeric_features),
    ('cat', categorical_pipeline, categorical_features),
    ('sex', sex_pipeline, sex_feature)
])

# Final Pipeline

In [12]:
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LogisticRegression())
])

In [13]:
pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['Age', 'Fare',
                                                   'FamilySize']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(drop='first',
                                                                                 handle_unknown='ignore'))]),
                                                  ['Pclass', 'Embarked']),
                                                 ('sex',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OrdinalEncoder())]),
                                                  ['Sex'])])),
                ('model', LogisticRegression())])

In [14]:
y_pred = pipeline.predict(X_test)

In [15]:
y_pred

array([0, 0, 0, 1, 1, 0, 1, 1, 1, 1, 0, 1, 0, 1, 1, 1, 0, 0, 0, 0, 0, 1,
       0, 0, 1, 1, 0, 1, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       1, 0, 0, 1, 0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0,
       1, 0, 1, 1, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 1, 0,
       1, 1, 0, 0, 0, 1, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1,
       0, 1, 0, 1, 0, 1, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0,
       0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 1, 1, 0, 1, 1, 0, 0, 1, 1, 0,
       1, 0, 0, 0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0,
       1, 0, 0])

In [16]:
print("Accuracy: ", accuracy_score(y_test, y_pred))

Accuracy:  0.8100558659217877


# Save The Model

In [17]:
joblib.dump(pipeline, 'titanic_model.pkl')

['titanic_model.pkl']